In [1]:
import datetime

import matplotlib.lines as mlines
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

In [2]:
sns.set_theme(style="whitegrid", palette="deep")
task_palette = sns.color_palette("deep")
plt.rcParams["text.usetex"] = True
plt.rcParams.update({"figure.titlesize": "small"})
plt.rcParams.update({"axes.titlesize": "small"})
plt.rcParams.update({"axes.labelsize": "small"})
plt.rcParams.update({"ytick.labelsize": "small"})
plt.rcParams.update({"xtick.labelsize": "small"})
plt.rcParams.update({"legend.fontsize": "small"})

In [3]:
path = "./results/results-mcs_scale_bfs-2026-04-15_20-29-06.csv"
df = pd.read_csv(path, sep=";")
df

,scheduler,safe_oracles,unsafe_oracles,use_case,use_idlesim,taskset_file,taskset_position,U,Uv,nbt,EDFVD_test,probability_of_HI,min_period,max_period,timeout,is_safe,automaton_depth,visited_count,duration_ns
0,edfvd,[],[],"BFS, no oracle",False,outputs/20251022_051939-statespace-rtss-bfs.txt,6,0.5,0.500000,2.0,1.0,0.5,5.0,10.0,False,True,18,35,20630
1,edfvd,[],[],"BFS, no oracle",False,outputs/20251022_051939-statespace-rtss-bfs.txt,0,0.5,0.500000,2.0,1.0,0.5,5.0,10.0,False,True,16,30,20790
2,edfvd,[],[],"BFS, no oracle",False,outputs/20251022_051939-statespace-rtss-bfs.txt,2,0.5,0.500000,2.0,1.0,0.5,5.0,10.0,False,True,20,40,22500
3,edfvd,[],[],"BFS, no oracle",False,outputs/20251022_051939-statespace-rtss-bfs.txt,1,0.5,0.500000,2.0,1.0,0.5,5.0,10.0,False,True,24,40,26830
4,edfvd,[],[],"BFS, no oracle",False,outputs/20251022_051939-statespace-rtss-bfs.txt,5,0.5,0.500000,2.0,1.0,0.5,5.0,10.0,False,True,20,40,22499
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,edfvd,[],[],"BFS, no oracle",False,outputs/20251022_051939-statespace-rtss-bfs.txt,177,0.5,0.497134,5.0,1.0,0.5,5.0,20.0,False,True,652080,1241509,1140553517
496,edfvd,[],[],"BFS, no oracle",False,outputs/20251022_051939-statespace-rtss-bfs.txt,225,0.5,0.495493,6.0,1.0,0.5,5.0,20.0,False,True,511632,1031372,1058729133
497,edfvd,[],[],"BFS, no oracle",False,outputs/20251022_051939-statespace-rtss-bfs.txt,247,0.5,0.502175,6.0,1.0,0.5,5.0,30.0,False,True,464100,875985,907231973
498,edfvd,[],[],"BFS, no oracle",False,outputs/20251022_051939-statespace-rtss-bfs.txt,246,0.5,0.498674,6.0,1.0,0.5,5.0,30.0,False,True,910455,1484767,1664115142


In [5]:
df["duration_s"] = df["duration_ns"] / 1e9

In [6]:
df["search_type"] = df["use_idlesim"].apply(lambda x: "ACBFS" if x else "BFS")

In [7]:
df = df.dropna(how="any")

In [8]:
s = df.shape[0] * [True]
df_plot = df.loc[s]
df_piv = df_plot.copy().pivot(
    index="taskset_position",
    columns="search_type",
    values=["duration_s", "visited_count", "max_period", "nbt"],
)
df_piv.columns = list(map(lambda x: "_".join(x), df_piv.columns))

In [9]:
df_piv = df_piv.dropna(how="any")

In [10]:
df_piv["Number of tasks"] = df_piv["nbt_BFS"].astype(int)
df_piv["Period max"] = df_piv["max_period_ACBFS"].astype(int)
df_piv["n"] = df_piv["nbt_BFS"].astype(int)
df_piv["T max"] = df_piv["max_period_ACBFS"].astype(int)

In [11]:
f, ax = plt.subplots(figsize=(6, 3))

sns.scatterplot(
    data=df_piv,
    x="duration_s_BFS",
    y="duration_s_ACBFS",
    style="T max",
    hue="n",
    ax=ax,
    palette=task_palette,
)
ax.set(xscale="log", yscale="log")
# ax.axis("equal")

lims = [
    10**-5.1,
    np.max([ax.get_xlim(), ax.get_ylim()]),  # max of both axes
]

# now plot both limits against eachother
ax.plot(lims, lims, "k--", alpha=0.75, zorder=1)

ax.set(xlim=lims, ylim=lims)

ax.set_xlabel("BFS")
ax.set_ylabel("ACBFS")
ax.set_title("Execution time (s)")

# sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1.1))

f.suptitle(r"$P_{\mathsf{HI}} = 0.5, T^{\mathsf{min}} = 5, U^{*} = 0.5$", y=1.02)

h, l = ax.get_legend_handles_labels()
ax.legend_.remove()
leg = [
    "$n$",
    "2",
    "3",
    "4",
    "5",
    "6",
    "$T^{\mathsf{max}}$",
    "10",
    "15",
    "20",
    "25",
    "30",
]
f.legend(h, leg, ncol=2, loc="upper left", bbox_to_anchor=(0.15, 0.85), framealpha=1)


x = np.log10(df_piv.sort_values(by="duration_s_BFS")["duration_s_BFS"])
y = np.log10(df_piv.sort_values(by="duration_s_BFS")["duration_s_ACBFS"])
m, b = np.polyfit(x, y, 1)
x_reg = np.linspace(lims, 100)
x_reg = np.log10(x_reg)
plt.plot(10**x_reg, 10 ** (m * x_reg + b), color="k", ls=":")

<>:42: SyntaxWarning: invalid escape sequence '\m'
<>:42: SyntaxWarning: invalid escape sequence '\m'
/tmp/ipykernel_153225/2994315444.py:42: SyntaxWarning: invalid escape sequence '\m'
  "$T^{\mathsf{max}}$",
/tmp/ipykernel_153225/2994315444.py:3: UserWarning: The palette list has more values (10) than needed (5), which may not be intended.
  sns.scatterplot(


In [12]:
f.savefig(
    f"./figs/{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}_bfs_acbfs.pdf",
    bbox_inches="tight",
)
f.savefig(
    "./figs/bfs_acbfs.pdf",
    bbox_inches="tight",
)

In [13]:
path = "./results/results-mcs_scale01-2026-04-15_20-29-48.csv"

df_sp = pd.read_csv(path, sep=";")
df_sp

,scheduler,safe_oracles,use_case,use_idlesim,unsafe_oracles,taskset_file,taskset_position,U,Uv,nbt,EDFVD_test,probability_of_HI,min_period,max_period,timeout,is_safe,automaton_depth,visited_count,duration_ns
0,edfvd,[],"ACBFS, no oracle",True,[],outputs/20260415_202915-statespace-rtss-utilis...,0,0.3,0.300000,5.0,1.0,0.5,5.0,30.0,False,True,9,19,50709
1,edfvd,[],"ACBFS, no oracle",True,[],outputs/20260415_202915-statespace-rtss-utilis...,4,0.3,0.295000,5.0,1.0,0.5,5.0,30.0,False,True,112,307,394065
2,edfvd,[],"ACBFS, no oracle",True,[],outputs/20260415_202915-statespace-rtss-utilis...,2,0.3,0.300000,5.0,1.0,0.5,5.0,30.0,False,True,116,321,393675
3,edfvd,[],"ACBFS, no oracle",True,[],outputs/20260415_202915-statespace-rtss-utilis...,3,0.3,0.295000,5.0,1.0,0.5,5.0,30.0,False,True,124,515,764190
4,edfvd,[],"ACBFS, no oracle",True,[],outputs/20260415_202915-statespace-rtss-utilis...,14,0.3,0.300000,5.0,1.0,0.5,5.0,30.0,False,True,150,1236,1698959
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
955,edfvd,[],"ACBFS, no oracle",True,[],outputs/20251022_053527-statespace-rtss-n-task...,388,0.5,0.495669,9.0,1.0,0.5,5.0,30.0,False,True,35,52,89659
956,edfvd,[],"ACBFS, no oracle",True,[],outputs/20251022_053527-statespace-rtss-n-task...,398,0.5,0.499886,9.0,1.0,0.5,5.0,30.0,False,True,50,56,87729
957,edfvd,[],"ACBFS, no oracle",True,[],outputs/20251022_053527-statespace-rtss-n-task...,397,0.5,0.504809,9.0,1.0,0.5,5.0,30.0,False,True,31,57,95339
958,edfvd,[],"ACBFS, no oracle",True,[],outputs/20251022_053527-statespace-rtss-n-task...,395,0.5,0.495972,9.0,1.0,0.5,5.0,30.0,False,True,44,63,92599


In [14]:
df_sp["taskset_file"].value_counts()

taskset_file
outputs/20251022_053515-statespace-rtss-period-max.txt     400
outputs/20251022_053527-statespace-rtss-n-tasks.txt        400
outputs/20260415_202915-statespace-rtss-utilisation.txt    160
Name: count, dtype: int64

In [15]:
df_sp.groupby("taskset_file")["taskset_position"].max()

taskset_file
outputs/20251022_053515-statespace-rtss-period-max.txt     399
outputs/20251022_053527-statespace-rtss-n-tasks.txt        399
outputs/20260415_202915-statespace-rtss-utilisation.txt    159
Name: taskset_position, dtype: int64

In [16]:
df_sp["duration_s"] = df_sp["duration_ns"] / 1e9

In [17]:
df_sp["timeout_int"] = df_sp["timeout"].astype(int)
df_sp.groupby(["nbt"])["timeout_int"].agg(["count", "sum"])

,count,sum
nbt,,
2.0,50,0
3.0,50,0
4.0,50,0
5.0,610,0
6.0,50,0
7.0,50,0
8.0,50,0
9.0,50,0


In [18]:
df_sp.groupby("nbt")["duration_s"].describe()

,count,mean,std,min,25%,50%,75%,max
nbt,,,,,,,,
2.0,50.0,0.000052,0.000020,0.000018,0.000037,0.000052,0.000065,0.000105
3.0,50.0,0.000084,0.000035,0.000032,0.000058,0.000077,0.000106,0.000160
4.0,50.0,0.000090,0.000056,0.000030,0.000062,0.000078,0.000095,0.000387
5.0,610.0,0.002246,0.005078,0.000023,0.000203,0.000614,0.002294,0.074077
6.0,50.0,0.000116,0.000059,0.000044,0.000077,0.000096,0.000141,0.000323
7.0,50.0,0.000117,0.000094,0.000031,0.000077,0.000101,0.000129,0.000701
8.0,50.0,0.000118,0.000063,0.000048,0.000084,0.000101,0.000132,0.000416
9.0,50.0,0.000126,0.000076,0.000027,0.000090,0.000105,0.000130,0.000506


In [20]:
# f, axs = plt.subplots(2, 1, figsize=(3.5*52/98*2, 3.5), gridspec_kw={"hspace": 0.25})
f, axs = plt.subplots(
    2, 1, figsize=(3.2 / 1.05, 3.2), gridspec_kw={"hspace": 0.25}, sharey=True
)


ax1 = axs[0]
ax2 = ax1.twinx()

s = df_sp["taskset_file"].str.contains("task")
dimension = "nbt"

# s = df_sp["taskset_file"].str.contains("period")
# dimension = "max_period"

s &= df_sp["use_case"] == "ACBFS, no oracle"


n_xp = 50

x_val = np.sort(df_sp.loc[s, dimension].unique())
x_val = list(map(int, x_val))
y_val_line = []
y_val_bar = []
for x in x_val:
    y_line = df_sp.loc[s & (df_sp[dimension] == x), "duration_s"].mean()
    y_val_line.append(y_line)

    y_bar = df_sp.loc[s & (df_sp[dimension] == x), "timeout_int"].sum()
    y_bar += n_xp - df_sp.loc[s & (df_sp[dimension] == x), "timeout_int"].count()
    y_val_bar.append(y_bar)

sns.lineplot(
    x=x_val,
    y=y_val_line,
    ax=ax1,
    marker="o",
    ms=9,
    estimator="median",
)
ax1.set(yscale="log")

ax2.bar(
    x=x_val,
    height=y_val_bar,
    color="w",
    edgecolor=task_palette[1],
    hatch="//",
    alpha=1,
    width=(x_val[1] - x_val[0]) * 0.8,
)

ax2.grid(False)
ax2.set_ylim(0, n_xp)


ax1.set_xlabel("Number of tasks ($n$), $T^{\mathsf{max}} = 30$")
ax1.set_ylabel("Execution time (s)")
ax2.set_ylabel("Timeouts")
ax1.minorticks_off()

ax1.xaxis.tick_top()
ax1.xaxis.set_label_position("top")
ax1.xaxis.set_ticks_position("none")
ax1.yaxis.set_ticks_position("none")
ax2.yaxis.set_ticks_position("none")
ax1.tick_params(axis="x", which="major", pad=0)


ax1 = axs[1]
ax2 = ax1.twinx()

# s = df_sp["taskset_file"].str.contains("task")
# dimension = "nbt"

s = df_sp["taskset_file"].str.contains("period")
dimension = "max_period"

s &= df_sp["use_case"] == "ACBFS, no oracle"


x_val = np.sort(df_sp.loc[s, dimension].unique())
x_val = list(map(int, x_val))
y_val_line = []
y_val_bar = []
for x in x_val:
    y_line = df_sp.loc[s & (df_sp[dimension] == x), "duration_s"].mean()
    y_val_line.append(y_line)

    y_bar = df_sp.loc[s & (df_sp[dimension] == x), "timeout_int"].sum()
    y_bar += n_xp - df_sp.loc[s & (df_sp[dimension] == x), "timeout_int"].count()
    y_val_bar.append(y_bar)

sns.lineplot(
    x=x_val,
    y=y_val_line,
    ax=ax1,
    marker="o",
    ms=9,
    estimator="median",
)
ax1.set(yscale="log")

ax2.bar(
    x=x_val,
    height=y_val_bar,
    color="w",
    edgecolor=task_palette[1],
    hatch="//",
    alpha=1,
    width=(x_val[1] - x_val[0]) * 0.8,
)

ax2.grid(False)
ax2.set_ylim(0, n_xp)


ax1.set_xlabel("Period max ($T^{\mathsf{max}}$), $n = 5$")
ax1.set_ylabel("Execution time (s)", labelpad=10)
ax2.set_ylabel("Timeouts")
ax1.minorticks_off()
ax1.yaxis.set_ticks_position("none")
ax2.yaxis.set_ticks_position("none")

f.suptitle(
    r"$P_{\mathsf{HI}} = 0.5, T^{\mathsf{min}} = 5, U^{*} = 0.5$",
    # x=0.55,
    # y=0.96,
    y=1.05,
)

blue_line = mlines.Line2D(
    [],
    [],
    color=task_palette[0],
    marker="o",
    linestyle="-",
    markeredgecolor="w",
    label="Execution time (s)",
    ms=9,
)

orange_box = mpatches.Patch(
    hatch="//", label="Timeouts", edgecolor=task_palette[1], facecolor="white"
)

new_handles = [blue_line, orange_box]


ax1.legend(
    handles=new_handles,
    loc="upper center",
    ncol=2,
    framealpha=0,
    columnspacing=0.8,
    bbox_to_anchor=(0.5, 1.28),
)

ax1.get_ylim()


f.tight_layout()

<>:57: SyntaxWarning: invalid escape sequence '\m'
<>:118: SyntaxWarning: invalid escape sequence '\m'
<>:57: SyntaxWarning: invalid escape sequence '\m'
<>:118: SyntaxWarning: invalid escape sequence '\m'
/tmp/ipykernel_153225/598293654.py:57: SyntaxWarning: invalid escape sequence '\m'
  ax1.set_xlabel("Number of tasks ($n$), $T^{\mathsf{max}} = 30$")
/tmp/ipykernel_153225/598293654.py:118: SyntaxWarning: invalid escape sequence '\m'
  ax1.set_xlabel("Period max ($T^{\mathsf{max}}$), $n = 5$")
/tmp/ipykernel_153225/598293654.py:162: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  f.tight_layout()


In [21]:
f.savefig(
    f"./figs/{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}_tmax_ntask.pdf",
    bbox_inches="tight",
)
f.savefig(
    "./figs/tmax_ntask.pdf",
    bbox_inches="tight",
)

In [3]:
path = "./results/results-mcs_schedulability-2026-04-29_12-10-16.csv"
df_sch = pd.read_csv(path, sep=";")
df_sch

,use_idlesim,safe_oracles,unsafe_oracles,periodic_tweak,use_case,scheduler,taskset_file,taskset_position,U,Uv,nbt,EDFVD_test,probability_of_HI,min_period,max_period,timeout,is_safe,automaton_depth,visited_count,duration_ns
0,True,[],['hi-over-demand'],False,EDF-VD (exact),edfvd,outputs/20251022_055741-scheduling-rtss.txt,3,0.50,0.499537,5.0,1.0,0.5,5.0,30.0,False,True,9,1806,4808688
1,True,[],['hi-over-demand'],False,EDF-VD (exact),edfvd,outputs/20251022_055741-scheduling-rtss.txt,2,0.50,0.497348,5.0,1.0,0.5,5.0,30.0,False,True,23,8483,28706269
2,True,[],['hi-over-demand'],False,EDF-VD (exact),edfvd,outputs/20251022_055741-scheduling-rtss.txt,7,0.50,0.499200,5.0,1.0,0.5,5.0,30.0,False,True,12,16631,56731735
3,True,[],['hi-over-demand'],False,EDF-VD (exact),edfvd,outputs/20251022_055741-scheduling-rtss.txt,0,0.50,0.502598,5.0,1.0,0.5,5.0,30.0,False,True,21,17980,57707294
4,True,[],['hi-over-demand'],False,EDF-VD (exact),edfvd,outputs/20251022_055741-scheduling-rtss.txt,11,0.50,0.500595,5.0,1.0,0.5,5.0,30.0,False,True,16,677,2049148
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21995,False,[],['hi-over-demand'],True,EDF-VD (pf),edfvd,outputs/20251022_055741-scheduling-rtss.txt,9654,0.95,0.946787,5.0,0.0,0.5,5.0,30.0,False,True,1045044,12191898,18419287520
21996,False,[],['hi-over-demand'],True,EDF-VD (pf),edfvd,outputs/20251022_055741-scheduling-rtss.txt,9698,0.95,0.946047,5.0,0.0,0.5,5.0,30.0,False,True,1863540,13469760,19332325037
21997,False,[],['hi-over-demand'],True,EDF-VD (pf),edfvd,outputs/20251022_055741-scheduling-rtss.txt,9200,0.95,0.946706,5.0,0.0,0.5,5.0,30.0,False,True,2034900,24348789,34246956250
21998,False,[],['hi-over-demand'],True,EDF-VD (pf),edfvd,outputs/20251022_055741-scheduling-rtss.txt,9813,0.95,0.949090,5.0,0.0,0.5,5.0,30.0,False,True,2960958,23766499,29711307035


In [4]:
path2 = "./results/results-mcs_schedulability-2026-05-01_11-58-31.csv"
df_sch2 = pd.read_csv(path2, sep=";")
df_sch2

,use_idlesim,safe_oracles,unsafe_oracles,periodic_tweak,use_case,scheduler,taskset_file,taskset_position,U,Uv,nbt,EDFVD_test,probability_of_HI,min_period,max_period,timeout,is_safe,automaton_depth,visited_count,duration_ns
0,False,[],['hi-over-demand'],True,EDF-VD (pf),edfvd,outputs/20251022_055741-scheduling-rtss.txt,11,0.50,0.500595,5.0,1.0,0.5,5.0,30.0,False,True,840.0,24394.0,6.787480e+07
1,False,[],['hi-over-demand'],True,EDF-VD (pf),edfvd,outputs/20251022_055741-scheduling-rtss.txt,2,0.50,0.497348,5.0,1.0,0.5,5.0,30.0,False,True,1320.0,102409.0,7.717792e+08
2,False,[],['hi-over-demand'],True,EDF-VD (pf),edfvd,outputs/20251022_055741-scheduling-rtss.txt,6,0.50,0.496329,5.0,1.0,0.5,5.0,30.0,False,True,5040.0,140363.0,1.119609e+09
3,False,[],['hi-over-demand'],True,EDF-VD (pf),edfvd,outputs/20251022_055741-scheduling-rtss.txt,34,0.50,0.496281,5.0,1.0,0.5,5.0,30.0,False,True,62790.0,91640.0,1.525026e+08
4,False,[],['hi-over-demand'],True,EDF-VD (pf),edfvd,outputs/20251022_055741-scheduling-rtss.txt,23,0.50,0.503968,5.0,1.0,0.5,5.0,30.0,False,True,252.0,226179.0,1.592270e+09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10995,False,[],['hi-over-demand'],True,EDF-VD (pf),edfvd,outputs/20251022_055741-scheduling-rtss.txt,9845,0.95,0.945074,5.0,0.0,0.5,5.0,30.0,True,NaN,NaN,NaN,NaN
10996,False,[],['hi-over-demand'],True,EDF-VD (pf),edfvd,outputs/20251022_055741-scheduling-rtss.txt,10260,1.00,0.997944,5.0,0.0,0.5,5.0,30.0,False,True,4620.0,32983806.0,6.895975e+11
10997,False,[],['hi-over-demand'],True,EDF-VD (pf),edfvd,outputs/20251022_055741-scheduling-rtss.txt,9876,0.95,0.950621,5.0,0.0,0.5,5.0,30.0,True,NaN,NaN,NaN,NaN
10998,False,[],['hi-over-demand'],True,EDF-VD (pf),edfvd,outputs/20251022_055741-scheduling-rtss.txt,9918,0.95,0.951288,5.0,0.0,0.5,5.0,30.0,True,NaN,NaN,NaN,NaN


In [5]:
df_sch["schedulable"] = df_sch["is_safe"]
df_sch2["schedulable"] = df_sch2["is_safe"]

In [6]:
# Filter df_sch for EDF-VD (exact)
df_sch = df_sch[df_sch["use_case"] == "EDF-VD (exact)"]
# Filter df_sch2 for EDF-VD (pf)
df_sch2 = df_sch2[df_sch2["use_case"] == "EDF-VD (pf)"]
# Merge the two forming a new df_sch
df_sch = pd.concat([df_sch, df_sch2], ignore_index=True)
df_sch

,use_idlesim,safe_oracles,unsafe_oracles,periodic_tweak,use_case,scheduler,taskset_file,taskset_position,U,Uv,...,EDFVD_test,probability_of_HI,min_period,max_period,timeout,is_safe,automaton_depth,visited_count,duration_ns,schedulable
0,True,[],['hi-over-demand'],False,EDF-VD (exact),edfvd,outputs/20251022_055741-scheduling-rtss.txt,3,0.50,0.499537,...,1.0,0.5,5.0,30.0,False,True,9.0,1806.0,4.808688e+06,True
1,True,[],['hi-over-demand'],False,EDF-VD (exact),edfvd,outputs/20251022_055741-scheduling-rtss.txt,2,0.50,0.497348,...,1.0,0.5,5.0,30.0,False,True,23.0,8483.0,2.870627e+07,True
2,True,[],['hi-over-demand'],False,EDF-VD (exact),edfvd,outputs/20251022_055741-scheduling-rtss.txt,7,0.50,0.499200,...,1.0,0.5,5.0,30.0,False,True,12.0,16631.0,5.673174e+07,True
3,True,[],['hi-over-demand'],False,EDF-VD (exact),edfvd,outputs/20251022_055741-scheduling-rtss.txt,0,0.50,0.502598,...,1.0,0.5,5.0,30.0,False,True,21.0,17980.0,5.770729e+07,True
4,True,[],['hi-over-demand'],False,EDF-VD (exact),edfvd,outputs/20251022_055741-scheduling-rtss.txt,11,0.50,0.500595,...,1.0,0.5,5.0,30.0,False,True,16.0,677.0,2.049148e+06,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21995,False,[],['hi-over-demand'],True,EDF-VD (pf),edfvd,outputs/20251022_055741-scheduling-rtss.txt,9845,0.95,0.945074,...,0.0,0.5,5.0,30.0,True,NaN,NaN,NaN,NaN,NaN
21996,False,[],['hi-over-demand'],True,EDF-VD (pf),edfvd,outputs/20251022_055741-scheduling-rtss.txt,10260,1.00,0.997944,...,0.0,0.5,5.0,30.0,False,True,4620.0,32983806.0,6.895975e+11,True
21997,False,[],['hi-over-demand'],True,EDF-VD (pf),edfvd,outputs/20251022_055741-scheduling-rtss.txt,9876,0.95,0.950621,...,0.0,0.5,5.0,30.0,True,NaN,NaN,NaN,NaN,NaN
21998,False,[],['hi-over-demand'],True,EDF-VD (pf),edfvd,outputs/20251022_055741-scheduling-rtss.txt,9918,0.95,0.951288,...,0.0,0.5,5.0,30.0,True,NaN,NaN,NaN,NaN,NaN


In [7]:
df_edfvd_test = df_sch[["taskset_position", "EDFVD_test", "U"]]
df_edfvd_test = df_edfvd_test.rename(columns={"EDFVD_test": "EDF-VD (sufficient)"})
df_edfvd_test["EDF-VD (sufficient)"] = df_edfvd_test["EDF-VD (sufficient)"].astype(bool)

In [8]:
d1 = df_sch[["taskset_position", "use_case", "schedulable", "U"]]
d2 = df_edfvd_test.melt(
    id_vars=["taskset_position", "U"], var_name="use_case", value_name="schedulable"
)

In [9]:
df_sched_comb = pd.concat([d1, d2], ignore_index=True)

In [10]:
# f, ax = plt.subplots(figsize=(3.5 * 46 / 98 * 2, 3.5))
f, ax = plt.subplots(figsize=(3.5, 3.5))

col_order = ["EDF-VD (sufficient)", "EDF-VD (exact)", "LWLF (exact)", "EDF-VD (pf)", "LWLF (pf)"]

sns.lineplot(
    data=df_sched_comb,
    x="U",
    y="schedulable",
    hue="use_case",
    ax=ax,
    markers=True,
    errorbar=None,
    style="use_case",
    ms=9,
    hue_order=col_order,
    style_order=col_order,
)

ax.set_ylabel("Schedulability ratio")
ax.set_xlabel("Average utilisation ($U^*$)")

# f.suptitle(
#     r"$P_{\mathsf{HI}} = 0.5, T^{\mathsf{min}} = 5, T^{\mathsf{max}} = 30, n=5$",
#     x=0.55,
#     # y=0.7,
# )

ax.set_title(
    r"$P_{\mathsf{HI}} = 0.5, T^{\mathsf{min}} = 5, T^{\mathsf{max}} = 30, n=5$",
)

# handles, labels = ax.get_legend_handles_labels()
# ax.legend(handles=handles, labels=labels, framealpha=1, loc="lower left")
# ax.legend(
#     handles=handles,
#     labels=labels,
#     loc="upper center",
#     ncol=2,
#     framealpha=0,
#     columnspacing=0.8,
#     bbox_to_anchor=(0.5, 1.4),
# )
sns.move_legend(
    ax,
    loc="center left",
    # ncol=2,
    framealpha=1,
    # columnspacing=0.8,
    # bbox_to_anchor=(0.5, 1.8),
    title="",
)

f.tight_layout()

/home/ochremarsh/Documents/real-time-systems/mixed-criticality-graph-xp/.venv-host/lib/python3.12/site-packages/seaborn/relational.py:293: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sub_data = grouped.apply(agg, other).reset_index()
/home/ochremarsh/Documents/real-time-systems/mixed-criticality-graph-xp/.venv-host/lib/python3.12/site-packages/seaborn/relational.py:293: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
 

In [11]:
f.savefig(
    f"./figs/{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}_scheduling.pdf",
    bbox_inches="tight",
)
f.savefig(
    f"./figs/{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}_scheduling.png",
    bbox_inches="tight",
)
f.savefig(
    "./figs/scheduling.pdf",
    bbox_inches="tight",
)

In [12]:
f, ax = plt.subplots(figsize=(3.5, 3.5))

df_time = (
    df_sch.loc[:, ["U", "duration_ns", "use_case"]]
    .dropna(how="any")
    .assign(duration_s=lambda d: d["duration_ns"] / 1e9)
    .sort_values(["use_case", "U"])
)

sns.lineplot(
    data=df_time,
    x="U",
    y="duration_s",
    hue="use_case",
    style="use_case",
    marker="o",
    ax=ax,
)

ax.set_ylabel("Time taken (s)")
ax.set_xlabel("Average utilisation ($U^*$)")

ax.set_title(
    r"$P_{\mathsf{HI}} = 0.5, n=5$",
)


/home/ochremarsh/Documents/real-time-systems/mixed-criticality-graph-xp/.venv-host/lib/python3.12/site-packages/seaborn/relational.py:293: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sub_data = grouped.apply(agg, other).reset_index()
/home/ochremarsh/Documents/real-time-systems/mixed-criticality-graph-xp/.venv-host/lib/python3.12/site-packages/seaborn/relational.py:293: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
 

Text(0.5, 1.0, '$P_{\\mathsf{HI}} = 0.5, n=5$')

In [13]:
f.savefig(
    f"./figs/{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}_edfvd_exact_time_uavg.pdf",
    bbox_inches="tight",
)
f.savefig(
    f"./figs/{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}_edfvd_exact_time_uavg.png",
    bbox_inches="tight",
)
f.savefig(
    "./figs/edfvd_exact_time_uavg.pdf",
    bbox_inches="tight",
)